# 02 · Baseline: how good is Whisper at Telugu without training?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pavanadityav-ai/telugu-asr-whisper/blob/main/notebooks/02_baseline_whisper.ipynb)

**Goal:** measure pre-trained `whisper-small` on the 472 test clips, with no training. This is the number fine-tuning has to beat.

**Runtime:** GPU. *Runtime → Change runtime type → T4 GPU*, then *Run all*. Takes about 15–20 minutes.

In [ ]:
# Setup: install libraries, get the project code, connect Google Drive
!pip install -q -U datasets jiwer
import os, sys
if not os.path.isdir("telugu-asr-whisper"):
    !git clone -q https://github.com/pavanadityav-ai/telugu-asr-whisper
sys.path.insert(0, "telugu-asr-whisper")

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/telugu-asr-whisper"  # results are saved here
os.makedirs(OUT, exist_ok=True)
print("Saving results to:", OUT)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Go to Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from src.audio_utils import decode_audio, load_fleurs_telugu

test = load_fleurs_telugu("test")
arrays = [decode_audio(a) for a in test["audio"]]
refs = list(test["transcription"])
print(len(arrays), "test clips loaded")

## Run Whisper
We tell Whisper the language is Telugu (`language="te"`). Without this, it guesses the language and sometimes translates into English instead.

In [ ]:
MODEL = "openai/whisper-small"   # you can also try "openai/whisper-medium" (slower)
MODEL_LABEL = MODEL.split("/")[-1] + " (no fine-tuning)"
FILE_TAG = "baseline_" + MODEL.split("/")[-1]

from src.asr import load_whisper, transcribe

model, processor = load_whisper(MODEL)
preds = transcribe(model, processor, arrays, batch_size=16)

## Score it
- **WER** (word error rate): share of words that are wrong.
- **CER** (character error rate): the same, per character.

Both can go **above 100%** if the model writes many extra words. Our metric code is written from scratch in `src/metrics.py`; the cell also checks it against the `jiwer` library.

In [ ]:
import json
import jiwer
import pandas as pd
from src.metrics import cer, sentence_scores, wer
from src.text_utils import normalize_text, script_profile

R = [normalize_text(r) for r in refs]
H = [normalize_text(p) for p in preds]

scores = {"model": MODEL_LABEL, "wer": wer(R, H), "cer": cer(R, H), "n_clips": len(R)}

# Check our from-scratch metrics against the jiwer library
pairs = [(r, h) for r, h in zip(R, H) if r]
jr, jh = [p[0] for p in pairs], [p[1] for p in pairs]
assert abs(scores["wer"] - jiwer.wer(jr, jh)) < 1e-6, "WER does not match jiwer"
assert abs(scores["cer"] - jiwer.cer(jr, jh)) < 1e-6, "CER does not match jiwer"

print(f"{MODEL_LABEL}:  WER = {100 * scores['wer']:.1f}%   CER = {100 * scores['cer']:.1f}%")

rows = []
for clip_id, n, r, h in zip(test["id"], test["num_samples"], R, H):
    s = sentence_scores(r, h)
    rows.append({"id": clip_id, "seconds": round(n / 16_000, 2), "reference": r,
                 "prediction": h, "wer": s["wer"], "cer": s["cer"],
                 "telugu_share": script_profile(h)["telugu"]})
df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/{FILE_TAG}_test_predictions.csv", index=False)
with open(f"{OUT}/{FILE_TAG}_scores.json", "w") as f:
    json.dump(scores, f, indent=2)
print("Saved", f"{OUT}/{FILE_TAG}_test_predictions.csv")

In [ ]:
# Best, middle and worst sentence (by character error rate)
ranked = df.sort_values("cer").reset_index(drop=True)
for label, i in [("BEST", 0), ("MIDDLE", len(ranked) // 2), ("WORST", len(ranked) - 1)]:
    row = ranked.iloc[i]
    print(f"--- {label}  (CER {100 * row.cer:.0f}%)")
    print("REF :", row.reference)
    print("PRED:", row.prediction, "\n")

## Does Whisper even write Telugu script?

In [ ]:
mostly_telugu = (df.telugu_share > 0.5).mean()
print(f"Predictions written mostly in Telugu script: {100 * mostly_telugu:.0f}%")
df[df.telugu_share <= 0.5][["reference", "prediction"]].head()

## My notes

_Write 3–5 things **you** noticed here, in your own words. These notes become the analysis section of the README and your SOP._

1. 
2. 
3. 